# 04 · Carga de la capa Gold en el Lakehouse de Fabric

Crea en el Lakehouse las tablas Delta que usa la demo de **Capa Ontológica en Fabric** (WIF 2026), a partir de los Excel de `Data/Gold` del repositorio.

**Antes de ejecutar**
1. Lakehouse `dlh_eolicas_galicia` creado **con esquemas habilitados** y adjuntado a este notebook como lakehouse predeterminado.
2. Los 6 Excel de `Data/Gold` subidos a **Files** del Lakehouse (en cualquier subcarpeta).
3. Ejecutar todas las celdas (**Run all**).

**Resultado:** esquema `gold` con 8 tablas: `D_Instalaciones`, `D_Propietarios_Instalaciones`, `D_Provincias`, `D_Geografia`, `D_Clasificacion_Instalaciones`, `D_Contratistas`, `F_Generacion_Eolicas`, `F_Ventas_Eolicas`.

In [ ]:
# Configuración
BASE   = "/lakehouse/default/Files"        # busca los Excel de Gold en Files y subcarpetas
SCHEMA = "gold"                            # esquema destino en el Lakehouse (pon None si el Lakehouse no tiene esquemas)

In [ ]:
import os, glob, re, unicodedata
import pandas as pd

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii','ignore').decode().lower()

def find(base, pattern):
    """Busca un fichero por nombre sin importar tildes/mayúsculas (evita problemas de codificación de nombres)."""
    for p in glob.glob(os.path.join(base, '**', '*.xlsx'), recursive=True):
        if norm(pattern) in norm(os.path.basename(p)):
            return p
    raise FileNotFoundError(f'No encuentro "{pattern}" en {base}')

def clean_cols(df):
    df = df.loc[:, [c for c in df.columns if not str(c).startswith('Unnamed')]]
    new = []
    for c in df.columns:
        c2 = unicodedata.normalize('NFKD', str(c)).encode('ascii','ignore').decode()
        c2 = re.sub(r'[ ,;{}()\n\t=/]+', '_', c2).strip('_')
        new.append(c2)
    df.columns = new
    return df

def build_tables(base):
    t = {}
    maestro = clean_cols(pd.read_excel(find(base, 'MaestroInstalacionesGalicia')))
    maestro['Potencia_unitaria_kw'] = maestro['Potencia_unitaria_kw'].astype(str).replace('nan', None)
    t['D_Instalaciones'] = maestro
    t['D_Propietarios_Instalaciones'] = (maestro[['Id_Propietario','Propietario']]
        .drop_duplicates().sort_values('Id_Propietario').reset_index(drop=True))
    geo = clean_cols(pd.read_excel(find(base, 'D_Geografia')))
    t['D_Geografia'] = geo
    gal = geo[geo['Id_ccaa'] == 12]
    t['D_Provincias'] = pd.DataFrame({
        'id_provincia': gal['Id_provincia'].values,
        'cod-ca': gal['Id_ccaa'].values,
        'cod-prv': gal['Cod-prv'].values,
        'nombre_ca': gal['Nombre_ccaa'].values,
        'nombre_provincia': gal['Nombre_provincia'].values,
        'nombre_provincia_completo': gal['Nombre_provincia_completo'].values})
    t['D_Clasificacion_Instalaciones'] = clean_cols(pd.read_excel(find(base, 'Clasificacion Instalaciones')))
    t['D_Contratistas'] = clean_cols(pd.read_excel(find(base, 'EmpresasContratistas')))
    t['F_Generacion_Eolicas'] = clean_cols(pd.read_excel(find(base, 'Generacion_parques_eolicos')))
    t['F_Ventas_Eolicas'] = clean_cols(pd.read_excel(find(base, 'Ventas_parques_eolicos')))
    return t


In [ ]:
# Construir las tablas (pandas) y adaptar nombres a los usados en la ontología
tables = build_tables(BASE)

# Medidas en minúsculas y clave de ventas como en la demo (Id_Venta)
lower_cols = {c: c.lower() for c in ['Energia_generada_gwh','Energia_vendida_gwh','Potencia_instalada_mw',
                                      'Precio_mercado_mwh_euro','Ingresos_estimados_euro']}
tables['F_Generacion_Eolicas'] = tables['F_Generacion_Eolicas'].rename(columns=lower_cols)
tables['F_Ventas_Eolicas'] = tables['F_Ventas_Eolicas'].rename(columns=lower_cols).rename(columns={'Id_Ventas':'Id_Venta'})

for name, df in tables.items():
    print(f"{name:32s} {df.shape[0]:>6} filas · {df.shape[1]:>2} columnas")

In [ ]:
# Escribir como tablas Delta
import pandas as pd

if SCHEMA:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

def to_spark(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].astype(object).where(pd.notna(df[c]), None)
    return spark.createDataFrame(df)

for name, df in tables.items():
    target = f"{SCHEMA}.{name}" if SCHEMA else name
    (to_spark(df).write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    print("OK ->", target)

In [ ]:
# Comprobación rápida
prefix = f"{SCHEMA}." if SCHEMA else ""
for name in tables:
    n = spark.table(prefix + name).count()
    print(f"{name:32s} {n:>6} filas")

# Integridad: todas las instalaciones de los hechos existen en D_Instalaciones
for fact in ["F_Generacion_Eolicas", "F_Ventas_Eolicas"]:
    huerfanos = spark.sql(f"""
        SELECT COUNT(*) AS n FROM {prefix}{fact} f
        LEFT ANTI JOIN {prefix}D_Instalaciones d ON f.Id_Instalacion = d.Id_Instalacion""").first().n
    print(fact, "-> filas sin instalación:", huerfanos)